In [28]:
import io
import zipfile
from pathlib import Path
from urllib.parse import quote
from collections.abc import AsyncIterable

from gitsource import chunk_documents
import httpx
from minsearch import AppendableIndex, Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS
from pydantic_ai import Agent, RunUsage, AgentStreamEvent, RunContext
from pydantic_ai.capabilities import ProcessEventStream, Instrumentation
from pydantic_ai.messages import FunctionToolCallEvent
from pydantic_ai.models.instrumented import InstrumentationSettings
from tqdm.auto import tqdm

import invoke_llm_trace

In [29]:
PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

INSTRUCTIONS = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2-3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Pulumi, so you do not need to include the word "pulumi" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

In [30]:
def get_or_download_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
) -> bytes:
    """
    Return the Markdown documentation as a zip archive, reusing a local copy if one already exists, and downloading it otherwise.

    Args:
        repo_owner (str): The owner of the GitHub repository.
        repo_name (str): The name of the GitHub repository.
        branch_name (str): The branch to download from.
        docs_path (str): Path of the documentation folder inside the repository.
        save_dir (str | None): Local directory to reuse/save the files in. If
            None, the files are only returned as an in-memory zip.

    Returns:
        bytes: A zip archive containing only the Markdown files under `docs_path`.
    """

    def _get(doc_dir: str | None = None) -> bytes | None:
        """Build a zip from an already-downloaded local directory.

        Args:
            doc_dir (str | None): Directory that may already contain the files.

        Returns:
            bytes | None: A zip archive of the local files, or None if the
            directory is missing or empty.
        """
        if doc_dir is None:
            return None

        out_dir = Path(doc_dir)
        existing_files = sorted(out_dir.rglob("*")) if out_dir.exists() else []
        existing_files = [f for f in existing_files if f.is_file()]
        if not existing_files:
            return None

        print(f"Using already-downloaded documentation in '{doc_dir}' ({len(existing_files)} files)")

        buffer = io.BytesIO()
        with zipfile.ZipFile(buffer, "w") as archive:
            for file in existing_files:
                archive.write(file, file.relative_to(out_dir).as_posix())
        return buffer.getvalue()

    def _download() -> bytes:
        """Download only the Markdown documentation from GitHub.

        Returns:
            bytes: A zip archive containing only the Markdown files under
            `docs_path`.
        """
        api = "https://api.github.com"
        raw = "https://raw.githubusercontent.com"

        with httpx.Client(follow_redirects=True) as client:
            # Resolve the branch to its root tree SHA
            branch = client.get(f"{api}/repos/{repo_owner}/{repo_name}/branches/{branch_name}")
            branch.raise_for_status()
            root_tree_sha = branch.json()["commit"]["commit"]["tree"]["sha"]
            print(f"Retrieved root tree SHA: {root_tree_sha}")

            # Locate the docs subtree inside the root tree
            root_tree = client.get(f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{root_tree_sha}")
            root_tree.raise_for_status()
            docs_entry = next(
                (item for item in root_tree.json()["tree"] if item["path"] == docs_path),
                None,
            )
            if docs_entry is None:
                raise ValueError(f"Path '{docs_path}' not found in {repo_owner}/{repo_name}@{branch_name}")
            print(f"Retrieved docs subtree '{docs_path}' (SHA: {docs_entry['sha']})")

            # Recursively list every file under docs/
            docs_tree = client.get(
                f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{docs_entry['sha']}?recursive=1"
            )
            docs_tree.raise_for_status()
            print(f"Retrieved recursive tree for '{docs_path}'")

            # Collect the Markdown files to download
            markdown_files = [
                item["path"]
                for item in docs_tree.json().get("tree", [])
                if item.get("type") == "blob" and item["path"].endswith((".md", ".mdx"))
            ]
            print(f"Found {len(markdown_files)} Markdown files to download")

            # Download each Markdown file and package it into an in-memory zip
            buffer = io.BytesIO()
            with zipfile.ZipFile(buffer, "w") as archive:
                for path in tqdm(markdown_files, desc="Downloading docs", unit="file"):
                    file_url = f"{raw}/{repo_owner}/{repo_name}/{branch_name}/{quote(f'{docs_path}/{path}')}"
                    response = client.get(file_url)
                    response.raise_for_status()
                    archive.writestr(path, response.content)

        return buffer.getvalue()

    # Reuse the local copy if it already exists
    data = _get(save_dir)
    if data is not None:
        return data

    # Otherwise download and (optionally) save to disk
    data = _download()

    if save_dir is not None:
        out_dir = Path(save_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            num_files = len(zf.namelist())
            zf.extractall(out_dir)
        print(f"Extracted {num_files} files to '{save_dir}'")

    return data

In [31]:
def retrieve_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
):
    response_contents = get_or_download_documentation(
        repo_owner=repo_owner,
        repo_name=repo_name,
        branch_name=branch_name,
        docs_path=docs_path,
        save_dir=save_dir,
    )
    documents = []
    with zipfile.ZipFile(io.BytesIO(response_contents)) as zip_ref:
        for file_path in zip_ref.namelist():
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                doc = {
                    'content': content,
                    'filename': file_path
                }
                documents.append(doc)
    return documents

In [32]:
documents = retrieve_documentation(
    repo_owner="pulumi",
    repo_name="docs",
    branch_name="master",
    docs_path="content",
    save_dir="docs",
)
documents[:5]

Using already-downloaded documentation in 'docs' (2186 files)


[{'content': '---\ntitle: The infrastructure as code platform for the AI era – Pulumi\nmeta_desc: Pulumi is an open-source infrastructure as code platform that helps humans and agents build and manage cloud infrastructure with real programming languages.\nmeta_image: /images/home/home-meta.png\n\ninclude_organization_schema: true\n\nsections:\n  - type: hero\n    layout: split\n    cta_secondary_text: Download open source\n    cta_secondary_link: /docs/install/\n    badge_highlight_text: "New:"\n    badge_text: "Native support for Terraform and HCL"\n    badge_link: /releases/terraform-state-backend-modules-hcl/\n    title: "*Unleash agents* <br>on your infrastructure."\n    description: |\n      Ship infrastructure faster with tools that let agents do what they do best — without letting them go off the rails.\n    anchor: hero\n    hero_animation: agent-loop\n\n  - type: logo_grid\n    title: Trusted by over 4,000 innovative companies\n    logos:\n      - snowflake\n      - nvidia\n  

In [33]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

In [34]:
index = AppendableIndex(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(document_chunks)

In [35]:
stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"pulumi"}

highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer="snowball", stop_words=stopwords),
)

In [36]:
file_index = {doc["filename"]: doc["content"] for doc in documents}
file_index["blog/10-things-you-can-do-with-neo/index.md"]

'---\ntitle: "10 Things You Can Do With Our Infrastructure Agent, Neo"\nfeature_image: feature.png\ndate: 2025-10-06\ndraft: false\nmeta_desc: "Discover 10 concrete ways Pulumi Neo helps automate infrastructure tasks while keeping humans in control."\nauthors:\n    - meagan-cojocar\ntags:\n    - ai\n    - platform-engineering\n    - pulumi-neo\n    - aws\nseries: neo-things\ncategory: product\n\nsocial:\n    twitter: "10 concrete workflows you can use Pulumi Neo, our new infrastructure agent, for: auto-fix AWS config violations, upgrade Lambda runtimes across accounts, generate infrastructure templates, respond to CVEs. Neo generates PRs, not production changes."\n    linkedin: "10 concrete workflows platform teams can use Pulumi Neo, our new infrastructure agent, for right now. From auto-remediating AWS config violations to upgrading Lambda runtimes across multiple accounts, Neo handles tasks that used to take days. Neo generates PRs, not one-way tickets. You stay in control while AI 

In [37]:
class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """

    def __init__(
        self, index: AppendableIndex, highlighter: Highlighter, file_index: dict[str, str],
    ) -> None:
        """
        Initialize the SearchTools instance.
        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (dict[str, str]): Mapping of filenames to file contents.
        """

        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> list[dict]:
        """
        Search the index for results matching a query and highlight them.
        Args:
            query (str): The search query to look up in the index.
        Returns:
            list[dict[str, Any]]: A list of highlighted search result objects.
        """

        search_results = self.index.search(query=query, num_results=5)
        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.
        Args:
            filename (str): The filename of the file to retrieve.
        Returns:
            str: The file contents if found, otherwise an error message.
        """

        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"

In [38]:
search_tools = SearchTools(index, highlighter, file_index)
tools = [search_tools.search, search_tools.get_file]

In [39]:
async def log_events(ctx: RunContext, events: AsyncIterable[AgentStreamEvent]) -> None:
    async for event in events:
        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            agent_name = ctx.agent.name if ctx.agent and hasattr(ctx.agent, "name") else "search"
            print(f"TOOL CALL ({agent_name}): {tool_name}({args})")

In [40]:
search_agent = Agent(
    name="search",
    model="anthropic:claude-opus-5",
    instructions=INSTRUCTIONS,
    tools=tools,
    capabilities=[
        Instrumentation(InstrumentationSettings(version=5)),
        ProcessEventStream(log_events),
    ],
)


In [41]:
query = (
    "AWS cross-account VPC peering connection between two accounts in different "
    "organizations, including route table routes and security groups, plus granting "
    "read-only access to an S3 bucket in the parent account via S3 bucket policy and cross-account IAM role. Show Pulumi code and steps."
)
result = await search_agent.run(query)

TOOL CALL (search): search({"query": "cross-account VPC peering connection between AWS accounts"})
TOOL CALL (search): search({"query": "aws provider assumeRole multiple accounts explicit provider configuration"})
TOOL CALL (search): search({"query": "S3 bucket policy IAM role read-only access"})
TOOL CALL (search): get_file({"filename": "blog/redis-cloud-provider/index.md"})
TOOL CALL (search): get_file({"filename": "docs/iac/concepts/providers/_index.md"})
TOOL CALL (search): get_file({"filename": "docs/iac/concepts/resources/options/provider.md"})
TOOL CALL (search): get_file({"filename": "blog/testable-iam-policy-documents/index.md"})


In [42]:
result.output

'## Iteration 3: Synthesis\n\n### What the knowledge base actually contains\n\n| Piece of your question | Covered in docs? | Source |\n|---|---|---|\n| Two AWS accounts in one program (explicit providers) | ✅ Fully documented | `docs/iac/concepts/providers/_index.md`, `docs/iac/concepts/resources/options/provider.md` |\n| VPC peering connection + accepter + routes + security group | ✅ Code shown (for AWS↔Redis Cloud peering, same account) | `blog/redis-cloud-provider/index.md` |\n| S3 bucket policy + cross-account IAM role (`sts:AssumeRole` from another account root) | ✅ Code shown | `blog/testable-iam-policy-documents/index.md` |\n| A dedicated *AWS-account-to-AWS-account* peering tutorial (`peerOwnerId`, peering across organizations) | ❌ **Not present** | — |\n\nI\'ll be explicit below about which lines come straight from the docs and which are my assembly of documented building blocks. **There is no document in this knowledge base that covers `peerOwnerId` / cross-organization peeri

In [43]:
messages = result.all_messages()

In [44]:
messages

[ModelRequest(parts=[UserPromptPart(content='AWS cross-account VPC peering connection between two accounts in different organizations, including route table routes and security groups, plus granting read-only access to an S3 bucket in the parent account via S3 bucket policy and cross-account IAM role. Show Pulumi code and steps.', timestamp=datetime.datetime(2026, 10, 13, 11, 49, 57, 552310, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 10, 13, 11, 49, 57, 552832, tzinfo=datetime.timezone.utc), instructions='You\'re a documentation assistant.\n\nAnswer the user question using only the documentation knowledge base.\n\nMake 3 iterations:\n\n1) First iteration:\n   - Perform one search using the search tool to identify potentially relevant documents.\n   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user question.\n\n2) Second iteration:\n   - Analyze the results from the previous search.\n   - Based on the filenames or documents returned

In [45]:
result.usage

RunUsage(cost=Decimal('0.336290'), details={'input_tokens': 41628, 'output_tokens': 5126, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'thinking_tokens': 1044}, input_tokens=41628, output_tokens=5126, requests=4, tool_calls=7)

In [46]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == "user-prompt":
            print("USER:", p.content)
        if part_kind == "tool-call":
            print("TOOL CALL:", p.tool_name, p.args)
        if part_kind == "tool-return":
            print("TOOL RETURN:", p.tool_name)
        if part_kind == "text":
            print(p.content)
    print()

request
USER: AWS cross-account VPC peering connection between two accounts in different organizations, including route table routes and security groups, plus granting read-only access to an S3 bucket in the parent account via S3 bucket policy and cross-account IAM role. Show Pulumi code and steps.

response
I'll research this step by step.

## Iteration 1: Initial search
TOOL CALL: search {"query": "cross-account VPC peering connection between AWS accounts"}

request
TOOL RETURN: search

response
**Reasoning for the first search:** The question combines two distinct AWS tasks — cross-account VPC peering and cross-account S3 access — so I started with the most specific concept ("cross-account VPC peering connection between AWS accounts") to find any document that covers multi-account networking. The results were mostly about Discovery & Governance cloud accounts and a Redis Cloud blog post about VPC peering, which suggests the knowledge base may not contain a dedicated networking/IAM t

In [47]:
new_prompt = "Create a reusable component for me to do this"
result2 = await search_agent.run(new_prompt, message_history=messages)

TOOL CALL (search): search({"query": "component resources authoring ComponentResource registerOutputs"})
TOOL CALL (search): search({"query": "ComponentResourceOptions providers map passing providers to child resources"})
TOOL CALL (search): search({"query": "component args interface inputs outputs parent option"})
TOOL CALL (search): get_file({"filename": "docs/iac/guides/building-extending/components/build-a-component.md"})
TOOL CALL (search): get_file({"filename": "docs/iac/guides/building-extending/components/when-to-build-a-component.md"})


In [48]:
result2.output

'## Iteration 3: Synthesis\n\n### One design decision has to be made first\n\nThe providers doc shows a component receiving providers as a **map keyed by package name**:\n\n```typescript\nnew MyResource("myResource", { providers: { aws: useast1, kubernetes: myk8s } });\n```\n\nThat map has a single `aws` key — so the standard `providers` resource option **cannot carry two AWS providers** into one component. Since your component needs both the parent and child accounts, the providers must come in another way. Two documented-supported options:\n\n1. **Pass provider instances as args** — `build-a-component.md` lists "Resource types" as supported arg types in TypeScript.\n2. **Pass role ARNs/region as args and construct both providers inside the component** — supported because "Explicit providers are themselves Pulumi resources, and their configuration values are Pulumi inputs."\n\nI\'ve used **option 2** below. It keeps the args to plain `Input<string>` values, which is what keeps the doo

In [49]:
def print_messages(messages: list):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == "user-prompt":
                print("USER:", p.content)
            if part_kind == "tool-call":
                print("TOOL CALL:", p.tool_name, p.args)
            if part_kind == "tool-return":
                print("TOOL RETURN:", p.tool_name)
            if part_kind == "text":
                print(p.content)
        print()

In [50]:
print_messages(result2.new_messages())

request
USER: Create a reusable component for me to do this

response
I'll research the component-authoring docs before writing this, since the component must correctly route resources to two different AWS providers.

## Iteration 1: Initial search
TOOL CALL: search {"query": "component resources authoring ComponentResource registerOutputs"}

request
TOOL RETURN: search

response
**Reasoning for the first search:** The user now wants the two-account peering + S3 access logic packaged as a reusable abstraction, so I searched for the core component-authoring API (`ComponentResource`, `registerOutputs`). This surfaced the dedicated component guides, which should tell me the required structure, the parenting rules, and how outputs must be registered.

## Iteration 2: Refining and retrieving documents
TOOL CALL: search {"query": "ComponentResourceOptions providers map passing providers to child resources"}
TOOL CALL: search {"query": "component args interface inputs outputs parent option"}


In [53]:
messages = []
usage = RunUsage()

while True:
    user_prompt = input("You:")
    if user_prompt.lower().strip() == "stop":
        break

    result = await search_agent.run(user_prompt, message_history=messages)
    usage = usage + result.usage

    print_messages(result.new_messages())
    messages.extend(result.new_messages())

TOOL CALL (search): search({"query": "VPC peering between two accounts"})
TOOL CALL (search): search({"query": "VPC peering connection requester accepter"})
TOOL CALL (search): search({"query": "multiple AWS accounts explicit provider assume role cross-account"})
TOOL CALL (search): get_file({"filename": "docs/clouds/aws/guides/vpc.md"})
TOOL CALL (search): get_file({"filename": "blog/redis-cloud-provider/index.md"})
TOOL CALL (search): get_file({"filename": "blog/organizing-aws-accounts-with-pulumi/index.md"})
request
USER: Add an entry on how to peer two VPC accounts 

response
I'll start by searching the knowledge base for existing VPC peering documentation.

**Iteration 1 — Initial search**

This query is appropriate because the user's request is about VPC peering between two accounts, so searching the exact topic term will reveal whether the knowledge base already contains peering guidance, related networking docs, or the structure/format I'd need to follow when adding a new entry

In [52]:
usage

RunUsage()